In [1]:
import numpy as np
import matplotlib.pyplot as plt
from keras.datasets import fashion_mnist
import wandb

# Load dataset
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()


29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [2]:
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

## Question 1
Download the fashion-MNIST dataset and plot 1 sample image for each class.

Use from keras.datasets import fashion_mnist for getting the fashion mnist dataset.

In [6]:
# Initialize wandb
wandb.init(project="DA6401-Assignment1", name="Backprop and gradient descent")

# Define a function to plot samples
def plot_samples_for_index(sample_index):
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    axes = axes.flatten()

    for class_id in range(10):
        # Find indices of samples belonging to this class
        class_indices = np.where(y_train == class_id)[0]

        # Make sure we don't exceed array bounds by using modulo
        safe_index = sample_index % len(class_indices)
        selected_idx = class_indices[safe_index]

        # Display the image
        axes[class_id].imshow(X_train[selected_idx], cmap='gray')
        axes[class_id].set_title(f"{class_names[class_id]}\nSample #{safe_index}")
        axes[class_id].axis('off')

    plt.tight_layout()
    return fig

# Create a custom panel in wandb to control the sample index
# Define a range of indices to explore
sample_indices = list(range(0, 35, 5))  # From 0 to 35, steps of 5

# Log multiple visualizations for different sample indices
for idx in sample_indices:
    fig = plot_samples_for_index(idx)

    # Log to wandb with the specific index as the step
    wandb.log({
        "Class Samples": wandb.Image(fig)
    }, step=idx)

    plt.close(fig)

# print("Finished logging sample visualizations to Wandb.")
wandb.finish()


## Question 2 :
Implement a feedforward neural network which takes images from the fashion-mnist data as input and outputs a probability distribution over the 10 classes.

Your code should be flexible such that it is easy to change the number of hidden layers and the number of neurons in each hidden layer.

In [7]:
class NeuralNetwork:
    def __init__(self, input_size=784, hidden_layers=[128, 64], output_size=10, activation='relu', init_method='xavier'):
        """
        Parameters:
        - input_size: Input dimension (default: 784 for 28x28 images)
        - hidden_layers: List specifying number of neurons in each hidden layer
        - output_size: Number of output classes (default: 10 for Fashion-MNIST)
        - activation: Activation function ('sigmoid', 'tanh', or 'relu' as these are mentioned in document)
        - init_method: Weight initialization method ('random' or 'xavier' as these are mentioned in document)
        """
        self.input_size = input_size
        self.hidden_layers = hidden_layers
        self.output_size = output_size
        self.activation = activation
        self.init_method = init_method

        # Architecture with input, hidden, and output layers
        layer_sizes = [input_size] + hidden_layers + [output_size]

        # Initialize weights and biases
        self.weights = []
        self.biases = []

        for i in range(len(layer_sizes) - 1):
            if init_method == 'xavier':
                # Xavier/Glorot initialization
                scale = np.sqrt(2.0 / (layer_sizes[i] + layer_sizes[i+1]))
                self.weights.append(np.random.randn(layer_sizes[i], layer_sizes[i+1]) * scale)
            else:
                # Simple random initialization
                self.weights.append(np.random.randn(layer_sizes[i], layer_sizes[i+1]) * 0.01)

            self.biases.append(np.zeros((1, layer_sizes[i+1])))

    def activate(self, h, derivative=False):
        """Apply activation function"""
        if self.activation == 'sigmoid':
            if derivative:
                return self.sigmoid(h) * (1 - self.sigmoid(h))
            return self.sigmoid(h)

        elif self.activation == 'tanh':
            if derivative:
                return 1 - np.power(self.tanh(h), 2)
            return self.tanh(h)

        elif self.activation == 'relu':
            if derivative:
                return np.where(h > 0, 1, 0)
            return np.maximum(0, h)

    def sigmoid(self, h):
        return 1 / (1 + np.exp(-np.clip(h, -500, 500)))  # Clip to avoid overflow

    def tanh(self, h):
        return np.tanh(h)

    def softmax(self, h):
        # Subtract max for numerical stability
        exp_h = np.exp(h - np.max(h, axis=1, keepdims=True))
        return exp_h / np.sum(exp_h, axis=1, keepdims=True)

    def forward(self, X):
        """
        Forward propagation

        Parameters:
        - X: Input data (batch_size x input_size)

        Returns:
        - Activations and pre-activations for backpropagation
        """
        # Store activations and pre-activations for backpropagation
        self.H = []  # Pre-activations
        self.A = [X]  # Activations (A[0] is the input)

        # Forward through hidden layers
        for i in range(len(self.weights) - 1):
            H = np.dot(self.A[i], self.weights[i]) + self.biases[i]
            self.H.append(H)
            self.A.append(self.activate(H))

        # Output layer with softmax
        H_out = np.dot(self.A[-1], self.weights[-1]) + self.biases[-1]
        self.H.append(H_out)
        output = self.softmax(H_out)
        self.A.append(output)

        return output

    def compute_loss(self, y_pred, y_true):
        """
        Compute cross-entropy loss

        Parameters:
        - y_pred: Predicted probabilities (softmax output)
        - y_true: One-hot encoded ground truth

        Returns:
        - Cross-entropy loss
        """
        m = y_true.shape[0]  # Batch size
        # Add small epsilon to avoid log(0)
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        cross_entropy = -np.sum(y_true * np.log(y_pred)) / m
        return cross_entropy

    def predict(self, X):
        """
        Make predictions

        Parameters:
        - X: Input data

        Returns:
        - Class predictions
        """
        output = self.forward(X)
        return np.argmax(output, axis=1)

## Question 3

Implement the backpropagation algorithm with support for the following optimisation functions

1. sgd
2. momentum based gradient descent
3. nesterov accelerated gradient descent
4. rmsprop
5. adam
6. nadam

In [8]:
class NeuralNetwork(NeuralNetwork):  # Extending the previous class
    def backpropagation(self, X, y, batch_size=128, learning_rate=0.001,
                        optimizer='sgd', epochs=10, beta1=0.9, beta2=0.999,
                        epsilon=1e-8, weight_decay=0):
        """
        Train the neural network using backpropagation

        Parameters:
        - X: Training data
        - y: Target labels (one-hot encoded)
        - batch_size: Size of mini-batches
        - learning_rate: Learning rate
        - optimizer: 'sgd', 'momentum', 'nag', 'rmsprop', 'adam', or 'nadam'
        - epochs: Number of training epochs
        - beta1: Exponential decay rate for first moment estimates (momentum)
        - beta2: Exponential decay rate for second moment estimates (RMSprop)
        - epsilon: Small constant for numerical stability
        - weight_decay: L2 regularization parameter

        Returns:
        - Training history (loss, accuracy)
        """
        m = X.shape[0]  # Number of training examples
        n_batches = int(np.ceil(m / batch_size))
        history = {'loss': [], 'accuracy': []}

        # Initialize=ing optimizer parameters
        if optimizer in ['momentum', 'nag']:
            # Velocity for momentum and NAG
            velocities_w = [np.zeros_like(w) for w in self.weights]
            velocities_b = [np.zeros_like(b) for b in self.biases]

        if optimizer in ['rmsprop', 'adam', 'nadam']:
            # Cache for RMSprop, Adam, and Nadam
            cache_w = [np.zeros_like(w) for w in self.weights]
            cache_b = [np.zeros_like(b) for b in self.biases]

        if optimizer in ['adam', 'nadam']:
            # Momentum for Adam and Nadam
            moment_w = [np.zeros_like(w) for w in self.weights]
            moment_b = [np.zeros_like(b) for b in self.biases]
            t = 0  # Timestep for bias correction

        for epoch in range(epochs):
            # Shuffle data for each epoch
            indices = np.random.permutation(m)
            X_shuffled = X[indices]
            y_shuffled = y[indices]

            epoch_loss = 0
            correct_preds = 0

            for batch in range(n_batches):
                # Get mini-batch
                start_idx = batch * batch_size
                end_idx = min((batch + 1) * batch_size, m)
                X_batch = X_shuffled[start_idx:end_idx]
                y_batch = y_shuffled[start_idx:end_idx]
                batch_size_actual = X_batch.shape[0]

                # Forward pass
                y_pred = self.forward(X_batch)
                loss = self.compute_loss(y_pred, y_batch)
                epoch_loss += loss * batch_size_actual

                # Calculate accuracy
                y_pred_classes = np.argmax(y_pred, axis=1)
                y_true_classes = np.argmax(y_batch, axis=1)
                correct_preds += np.sum(y_pred_classes == y_true_classes)

                # Backpropagation - compute gradients
                dA = y_pred - y_batch  # Derivative of softmax with cross-entropy

                dW = []
                db = []

                # Output layer gradients
                dW_last = np.dot(self.A[-2].T, dA) / batch_size_actual
                db_last = np.sum(dA, axis=0, keepdims=True) / batch_size_actual

                # Adding L2 regularization
                if weight_decay > 0:
                    dW_last += weight_decay * self.weights[-1]

                dW.insert(0, dW_last)
                db.insert(0, db_last)

                # Backpropagate through hidden layers
                dA_prev = dA
                for i in reversed(range(len(self.weights) - 1)):
                    # Compute dZ for the current layer
                    dZ = np.dot(dA_prev, self.weights[i+1].T) * self.activate(self.Z[i], derivative=True)

                    # Compute gradients
                    dW_curr = np.dot(self.A[i].T, dZ) / batch_size_actual
                    db_curr = np.sum(dZ, axis=0, keepdims=True) / batch_size_actual

                    # Add L2 regularization
                    if weight_decay > 0:
                        dW_curr += weight_decay * self.weights[i]

                    dW.insert(0, dW_curr)
                    db.insert(0, db_curr)

                    # Update dA for next iteration
                    dA_prev = dZ

                # Update weights and biases based on optimizer
                if optimizer == 'sgd':
                    # Standard SGD
                    for i in range(len(self.weights)):
                        self.weights[i] -= learning_rate * dW[i]
                        self.biases[i] -= learning_rate * db[i]

                elif optimizer == 'momentum':
                    # SGD with Momentum
                    for i in range(len(self.weights)):
                        velocities_w[i] = beta1 * velocities_w[i] + (1 - beta1) * dW[i]
                        velocities_b[i] = beta1 * velocities_b[i] + (1 - beta1) * db[i]

                        self.weights[i] -= learning_rate * velocities_w[i]
                        self.biases[i] -= learning_rate * velocities_b[i]

                elif optimizer == 'nag':
                    # Nesterov Accelerated Gradient
                    for i in range(len(self.weights)):
                        # Update velocity first
                        velocities_w[i] = beta1 * velocities_w[i] + (1 - beta1) * dW[i]
                        velocities_b[i] = beta1 * velocities_b[i] + (1 - beta1) * db[i]

                        # "Look ahead" - use the updated velocity for the gradient step
                        self.weights[i] -= learning_rate * (beta1 * velocities_w[i] + (1 - beta1) * dW[i])
                        self.biases[i] -= learning_rate * (beta1 * velocities_b[i] + (1 - beta1) * db[i])

                elif optimizer == 'rmsprop':
                    # RMSprop
                    for i in range(len(self.weights)):
                        # Update cache (moving average of squared gradients)
                        cache_w[i] = beta2 * cache_w[i] + (1 - beta2) * (dW[i] ** 2)
                        cache_b[i] = beta2 * cache_b[i] + (1 - beta2) * (db[i] ** 2)

                        # Update weights and biases
                        self.weights[i] -= learning_rate * dW[i] / (np.sqrt(cache_w[i]) + epsilon)
                        self.biases[i] -= learning_rate * db[i] / (np.sqrt(cache_b[i]) + epsilon)

                elif optimizer == 'adam':
                    # Adam optimizer
                    t += 1  # Increment timestep

                    for i in range(len(self.weights)):
                        # Update first moment estimate (momentum)
                        moment_w[i] = beta1 * moment_w[i] + (1 - beta1) * dW[i]
                        moment_b[i] = beta1 * moment_b[i] + (1 - beta1) * db[i]

                        # Update second moment estimate (RMSprop)
                        cache_w[i] = beta2 * cache_w[i] + (1 - beta2) * (dW[i] ** 2)
                        cache_b[i] = beta2 * cache_b[i] + (1 - beta2) * (db[i] ** 2)

                        # Bias correction
                        moment_w_corrected = moment_w[i] / (1 - beta1 ** t)
                        moment_b_corrected = moment_b[i] / (1 - beta1 ** t)
                        cache_w_corrected = cache_w[i] / (1 - beta2 ** t)
                        cache_b_corrected = cache_b[i] / (1 - beta2 ** t)

                        # Update weights and biases
                        self.weights[i] -= learning_rate * moment_w_corrected / (np.sqrt(cache_w_corrected) + epsilon)
                        self.biases[i] -= learning_rate * moment_b_corrected / (np.sqrt(cache_b_corrected) + epsilon)

                elif optimizer == 'nadam':
                    # Nadam (Adam with Nesterov momentum)
                    t += 1  # Increment timestep

                    for i in range(len(self.weights)):
                        # Update first moment estimate
                        moment_w[i] = beta1 * moment_w[i] + (1 - beta1) * dW[i]
                        moment_b[i] = beta1 * moment_b[i] + (1 - beta1) * db[i]

                        # Update second moment estimate
                        cache_w[i] = beta2 * cache_w[i] + (1 - beta2) * (dW[i] ** 2)
                        cache_b[i] = beta2 * cache_b[i] + (1 - beta2) * (db[i] ** 2)

                        # Bias correction
                        moment_w_corrected = moment_w[i] / (1 - beta1 ** t)
                        moment_b_corrected = moment_b[i] / (1 - beta1 ** t)
                        cache_w_corrected = cache_w[i] / (1 - beta2 ** t)
                        cache_b_corrected = cache_b[i] / (1 - beta2 ** t)

                        # Nesterov momentum update
                        moment_w_nesterov = beta1 * moment_w_corrected + (1 - beta1) * dW[i] / (1 - beta1 ** t)
                        moment_b_nesterov = beta1 * moment_b_corrected + (1 - beta1) * db[i] / (1 - beta1 ** t)

                        # Update weights and biases
                        self.weights[i] -= learning_rate * moment_w_nesterov / (np.sqrt(cache_w_corrected) + epsilon)
                        self.biases[i] -= learning_rate * moment_b_nesterov / (np.sqrt(cache_b_corrected) + epsilon)

            # Calculate epoch statistics
            epoch_loss /= m
            epoch_accuracy = correct_preds / m

            history['loss'].append(epoch_loss)
            history['accuracy'].append(epoch_accuracy)

            print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.4f}")

        return history